In [38]:
from pathlib import Path
import json
import copy
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models.video import (
    r2plus1d_18,
    R2Plus1D_18_Weights,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.0+cu130
CUDA available: True


In [39]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [40]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [41]:
PROJECT_ROOT = Path("../../").resolve()

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "echo"
    / "echonet"
)

MANIFEST_PATH = (
    PROCESSED_DIR
    / "echonet_manifest.csv"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "echo"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "metrics"
    / "echo"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Manifest:", MANIFEST_PATH)
print("Exists:", MANIFEST_PATH.exists())

Manifest: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\echo\echonet\echonet_manifest.csv
Exists: True


In [42]:
echo = pd.read_csv(
    MANIFEST_PATH
)

print("Manifest shape:", echo.shape)

display(
    echo.head()
)

Manifest shape: (1530, 12)


,video_name,video_path,EF,split,frame_count,fps,width,height,frame_indices,lv_dysfunction,reduced_lvef,hfrEF
0,0X100009310A3BD7FC,data\raw\echo\echonet\Videos\0X100009310A3BD7F...,78.498406,val,174,50.0,112,112,"[23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63, 6...",0,0,0
1,0X1002E8FBACD08477,data\raw\echo\echonet\Videos\0X1002E8FBACD0847...,59.101988,train,215,50.0,112,112,"[43, 47, 51, 55, 59, 63, 67, 71, 75, 79, 83, 8...",0,0,0
2,0X1005D03EED19C65B,data\raw\echo\echonet\Videos\0X1005D03EED19C65...,62.363798,train,104,50.0,112,112,"[0, 3, 7, 10, 13, 17, 20, 23, 27, 30, 33, 37, ...",0,0,0
3,0X10075961BC11C88E,data\raw\echo\echonet\Videos\0X10075961BC11C88...,54.545097,train,122,55.0,112,112,"[0, 4, 8, 12, 16, 20, 23, 27, 31, 35, 39, 43, ...",0,0,0
4,0X10094BA0A028EAC3,data\raw\echo\echonet\Videos\0X10094BA0A028EAC...,24.887742,val,207,52.0,112,112,"[39, 43, 47, 51, 55, 59, 63, 67, 71, 75, 79, 8...",1,1,1


In [43]:
required_columns = [
    "video_path",
    "EF",
    "split",
    "frame_indices",
]

missing = [
    col
    for col in required_columns
    if col not in echo.columns
]

if missing:
    raise ValueError(
        f"Missing manifest columns: {missing}"
    )

echo["EF"] = pd.to_numeric(
    echo["EF"],
    errors="coerce"
)

echo = echo[
    echo["EF"].between(0, 100)
].copy()

echo = echo[
    echo["split"].isin(
        ["train", "val", "test"]
    )
].copy()

echo = echo.reset_index(
    drop=True
)

print("Usable records:", len(echo))
print("\nSplit:")
print(echo["split"].value_counts())

Usable records: 1530

Split:
split
train    1144
val       204
test      182
Name: count, dtype: int64


In [44]:
NUM_FRAMES = 32
FRAME_STRIDE = 4

IMAGE_SIZE = 112

BATCH_SIZE = 4

NUM_WORKERS = 0

EPOCHS = 20

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

print(
    f"Clip: {NUM_FRAMES} frames, "
    f"stride={FRAME_STRIDE}"
)

print(
    f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}"
)

print("Batch size:", BATCH_SIZE)

Clip: 32 frames, stride=4
Image size: 112x112
Batch size: 4


In [45]:
IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)

In [46]:
class EchoNetDataset(Dataset):

    def __init__(
        self,
        dataframe,
        project_root,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.project_root = Path(
            project_root
        )

    def __len__(self):
        return len(self.df)

    def _load_clip(
        self,
        video_path,
        frame_indices,
    ):
        video_path = (
            self.project_root
            / video_path
        )

        cap = cv2.VideoCapture(
            str(video_path)
        )

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video: "
                f"{video_path}"
            )

        frames = []

        for frame_index in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_index)
            )

            ret, frame = cap.read()

            if not ret:
                cap.release()

                raise RuntimeError(
                    f"Could not read frame "
                    f"{frame_index} from "
                    f"{video_path}"
                )

            # BGR -> grayscale
            gray = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2GRAY
            )

            # Resize
            gray = cv2.resize(
                gray,
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE
                ),
                interpolation=cv2.INTER_AREA
            )

            # grayscale -> 3 channels
            rgb = np.stack(
                [gray, gray, gray],
                axis=-1
            )

            # uint8 -> [0, 1]
            rgb = (
                rgb.astype(np.float32)
                / 255.0
            )

            # ImageNet normalization
            rgb = (
                rgb - IMAGENET_MEAN
            ) / IMAGENET_STD

            frames.append(rgb)

        cap.release()

        # [T, H, W, C]
        clip = np.stack(
            frames,
            axis=0
        )

        # [C, T, H, W]
        clip = np.transpose(
            clip,
            (3, 0, 1, 2)
        )

        return torch.tensor(
            clip,
            dtype=torch.float32
        )

    def __getitem__(self, index):

        row = self.df.iloc[index]

        frame_indices = json.loads(
            row["frame_indices"]
        )

        # Ensure exactly 32 indices
        if len(frame_indices) != NUM_FRAMES:
            raise ValueError(
                f"Expected {NUM_FRAMES} frames, "
                f"got {len(frame_indices)}"
            )

        clip = self._load_clip(
            row["video_path"],
            frame_indices
        )

        ef = torch.tensor(
            float(row["EF"]),
            dtype=torch.float32
        )

        return clip, ef

In [47]:
train_df = echo[
    echo["split"] == "train"
].copy()

val_df = echo[
    echo["split"] == "val"
].copy()

test_df = echo[
    echo["split"] == "test"
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1144
Validation: 204
Test: 182


In [48]:
DEBUG_LIMIT = None
# Example:
# DEBUG_LIMIT = 50

if DEBUG_LIMIT is not None:

    train_df = train_df.head(
        DEBUG_LIMIT
    )

    val_df = val_df.head(
        min(DEBUG_LIMIT, len(val_df))
    )

    test_df = test_df.head(
        min(DEBUG_LIMIT, len(test_df))
    )

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

Training records: 1144
Validation records: 204
Test records: 182


In [49]:
train_dataset = EchoNetDataset(
    train_df,
    PROJECT_ROOT
)

val_dataset = EchoNetDataset(
    val_df,
    PROJECT_ROOT
)

test_dataset = EchoNetDataset(
    test_df,
    PROJECT_ROOT
)

print("Datasets created.")

Datasets created.


In [50]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("DataLoaders created.")

DataLoaders created.


In [51]:
sample_clip, sample_ef = train_dataset[0]

print("Clip shape:", sample_clip.shape)
print("EF:", sample_ef.item())
print("Dtype:", sample_clip.dtype)

Clip shape: torch.Size([3, 32, 112, 112])
EF: 59.10198974609375
Dtype: torch.float32


In [52]:
sample_batch, sample_targets = next(
    iter(train_loader)
)

print("Batch shape:", sample_batch.shape)
print("Target shape:", sample_targets.shape)
print("Targets:", sample_targets[:5])

Batch shape: torch.Size([4, 3, 32, 112, 112])
Target shape: torch.Size([4])
Targets: tensor([38.4642, 66.0317, 40.4093, 64.6616])


In [53]:
weights = (
    R2Plus1D_18_Weights.DEFAULT
)

model = r2plus1d_18(
    weights=weights
)

print(model)

VideoResNet(
  (stem): R2Plus1dStem(
    (0): Conv3d(3, 45, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
    (1): BatchNorm3d(45, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv3d(45, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
    (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv2Plus1D(
          (0): Conv3d(64, 144, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1), bias=False)
          (1): BatchNorm3d(144, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv3d(144, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
        )
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, aff

In [54]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    1
)

model = model.to(
    DEVICE
)

print(model.fc)

Linear(in_features=512, out_features=1, bias=True)


In [55]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [56]:
USE_AMP = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print("Mixed precision enabled:", USE_AMP)

Mixed precision enabled: True


In [57]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()

    running_loss = 0.0
    samples = 0

    for clips, targets in loader:

        clips = clips.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            outputs = model(
                clips
            ).squeeze(1)

            loss = criterion(
                outputs,
                targets
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        batch_size = clips.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )

        samples += batch_size

    return running_loss / samples

In [58]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
):
    model.eval()

    total_squared_error = 0.0
    total_absolute_error = 0.0
    total_samples = 0

    all_predictions = []
    all_targets = []

    for clips, targets in loader:

        clips = clips.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(
            clips
        ).squeeze(1)

        squared_error = (
            (outputs - targets) ** 2
        )

        absolute_error = (
            torch.abs(
                outputs - targets
            )
        )

        batch_size = clips.size(0)

        total_squared_error += (
            squared_error.sum().item()
        )

        total_absolute_error += (
            absolute_error.sum().item()
        )

        total_samples += batch_size

        all_predictions.extend(
            outputs.cpu().numpy().tolist()
        )

        all_targets.extend(
            targets.cpu().numpy().tolist()
        )

    mse = (
        total_squared_error
        / total_samples
    )

    mae = (
        total_absolute_error
        / total_samples
    )

    return {
        "mse": mse,
        "mae": mae,
        "predictions": np.array(
            all_predictions
        ),
        "targets": np.array(
            all_targets
        ),
    }

In [59]:
import time

EPOCHS = 20
PATIENCE = 5

best_val_mse = float("inf")
best_state = None
patience_counter = 0

history = []

training_start = time.time()

print("=" * 70)
print("Starting EchoNet R(2+1)D-18 training")
print("=" * 70)
print(f"Device: {DEVICE}")
print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print()

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Training started..."
    )

    train_mse = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
    )

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Training finished. "
        f"Running validation..."
    )

    val_results = evaluate(
        model,
        val_loader,
        criterion,
    )

    val_mse = val_results["mse"]
    val_mae = val_results["mae"]

    scheduler.step(
        val_mse
    )

    epoch_time = time.time() - epoch_start
    total_time = time.time() - training_start

    history.append({
        "epoch": epoch,
        "train_mse": train_mse,
        "val_mse": val_mse,
        "val_mae": val_mae,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "epoch_time_seconds": epoch_time,
    })

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Train MSE: {train_mse:.4f} | "
        f"Val MSE: {val_mse:.4f} | "
        f"Val MAE: {val_mae:.4f}"
    )

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Time: {epoch_time / 60:.2f} min | "
        f"Total: {total_time / 60:.2f} min"
    )

    if val_mse < best_val_mse:

        best_val_mse = val_mse

        best_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

        print(
            f"✓ New best model! "
            f"Validation MSE = {best_val_mse:.4f}"
        )

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print(
                "\nEarly stopping triggered."
            )

            break

    print("-" * 70)

total_training_time = time.time() - training_start

print("=" * 70)
print("Training complete")
print(
    f"Total training time: "
    f"{total_training_time / 60:.2f} minutes"
)
print(
    f"Best validation MSE: "
    f"{best_val_mse:.4f}"
)
print("=" * 70)

Starting EchoNet R(2+1)D-18 training
Device: cuda
Training samples: 1,144
Validation samples: 204
Epochs: 20
Batch size: 4

[Epoch 1/20] Training started...


KeyboardInterrupt: 

In [ ]:
if not history:
    raise RuntimeError(
        "Training history is empty. "
        "Run the training cell first."
    )

history_df = pd.DataFrame(
    history
)

display(history_df)

plt.figure(figsize=(8, 5))

plt.plot(
    history_df["epoch"],
    history_df["train_mse"],
    label="Train MSE"
)

plt.plot(
    history_df["epoch"],
    history_df["val_mse"],
    label="Validation MSE"
)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("EchoNet R(2+1)D-18 Training")

plt.legend()

plt.tight_layout()
plt.show()

RuntimeError: Training history is empty. Run the training cell first.

In [ ]:
if best_state is None:
    raise RuntimeError(
        "No best model state was recorded."
    )

model.load_state_dict(
    best_state
)

print(
    f"Best validation MSE: "
    f"{best_val_mse:.4f}"
)

RuntimeError: No best model state was recorded.

In [ ]:
test_results = evaluate(
    model,
    test_loader,
    criterion,
)

print(
    f"Test MSE : "
    f"{test_results['mse']:.4f}"
)

print(
    f"Test MAE : "
    f"{test_results['mae']:.4f}"
)

KeyboardInterrupt: 

In [ ]:
predictions = test_results[
    "predictions"
]

targets = test_results[
    "targets"
]

plt.figure(figsize=(7, 7))

plt.scatter(
    targets,
    predictions,
    alpha=0.5
)

minimum = min(
    targets.min(),
    predictions.min()
)

maximum = max(
    targets.max(),
    predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual EF (%)")
plt.ylabel("Predicted EF (%)")
plt.title("EchoNet EF: Actual vs Predicted")

plt.tight_layout()
plt.show()

NameError: name 'test_results' is not defined

In [ ]:
predicted_ef = predictions

lv_dysfunction = (
    predicted_ef < 50
)

hfrEF = (
    predicted_ef < 40
)

print(
    "Predicted EF < 50%:",
    lv_dysfunction.sum()
)

print(
    "Predicted EF < 40%:",
    hfrEF.sum()
)

NameError: name 'predictions' is not defined

In [ ]:
checkpoint_path = (
    CHECKPOINT_DIR
    / "echonet_r2plus1d18.pt"
)

checkpoint = {
    "model_state_dict": model.state_dict(),

    "architecture": "r2plus1d_18",

    "input_frames": NUM_FRAMES,

    "frame_stride": FRAME_STRIDE,

    "image_size": IMAGE_SIZE,

    "target": "EF",

    "loss": "MSELoss",

    "imagenet_mean": IMAGENET_MEAN.tolist(),

    "imagenet_std": IMAGENET_STD.tolist(),

    "best_val_mse": best_val_mse,
}

torch.save(
    checkpoint,
    checkpoint_path
)

print("Saved:")
print(checkpoint_path)

Saved:
D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\echo\echonet_r2plus1d18.pt


In [ ]:
history_df.to_csv(
    RESULTS_DIR
    / "training_history.csv",
    index=False
)

test_summary = pd.DataFrame([{
    "test_mse": test_results["mse"],
    "test_mae": test_results["mae"],
}])

test_summary.to_csv(
    RESULTS_DIR
    / "test_metrics.csv",
    index=False
)

print(
    "Results saved to:",
    RESULTS_DIR
)

NameError: name 'history_df' is not defined

In [ ]:
print("========================================")
print("EchoNet training complete")
print("========================================")

print(
    f"Architecture : R(2+1)D-18"
)

print(
    f"Input        : "
    f"{NUM_FRAMES} frames × "
    f"{IMAGE_SIZE}×{IMAGE_SIZE}"
)

print(
    f"Target       : EF (%)"
)

print(
    f"Best Val MSE : "
    f"{best_val_mse:.4f}"
)

print(
    f"Test MSE     : "
    f"{test_results['mse']:.4f}"
)

print(
    f"Test MAE     : "
    f"{test_results['mae']:.4f}"
)

print(
    f"Checkpoint   : "
    f"{checkpoint_path}"
)

EchoNet training complete
Architecture : R(2+1)D-18
Input        : 32 frames × 112×112
Target       : EF (%)
Best Val MSE : inf


NameError: name 'test_results' is not defined